In [0]:
df = spark.read.table("ecommerce_analytics.bronze.sales_orders")
df.display()


In [0]:
df.printSchema()

In [0]:
''' Clicked Item
customer_id |  product_id           |score
19476252    | AVpfPEx61cnluZ0-gyT9  |34
19476252    | AVpfuJ4pilAPnD_xhDyM  |98
19476252    | AVpe6jFBilAPnD_xQxO2  |60
'''

'''ordered products 
customer_id     curr            id                      name ......
19476252        USD            AVpfuJ4pilAPnD_xhDyM   Rony LBT-GPX555 Mini-System with Blue

'''


**-> Array of Array**
```
[
    ["AVpgIu4Q1cnluZ0-xBK-","13"],
    ["AVpfeG5oilAPnD_xcTsG","27"],
    ["AVqVGaEBv8e3D1O-ldFu","64"],
    ["AVpg-Wj61cnluZ0-8sZe","87"],
    ["AVphTO5W1cnluZ0-Aygg","52"],
    ["AVpfMVD-ilAPnD_xW6bu","49"]
]
```
**-> Array of Struct**
```
[
    {"curr":"USD","id":"AVpfuJ4pilAPnD_xhDyM","name":"Rony LBT-GPX555 Mini-System with Bluetooth and NFC","price":"993","promotion_info":null,"qty":"3","unit":"pcs"},
    {"curr":"USD","id":"AVpe6jFBilAPnD_xQxO2","name":"Aeon 71.5 x 130.9 16:9 Fixed Frame Projection Screen with CineWhite Projection Surface","price":"218","promotion_info":null,"qty":"3","unit":"pcs"},
    {"curr":"USD","id":"AVpfIODe1cnluZ0-eg35","name":"Cyber-shot DSC-WX220 Digital Camera (Black)","price":"448","promotion_info":null,"qty":"2","unit":"pcs"}
]
```
**-> Array of Struct**
```
[
    {"promo_disc":0.03,"promo_id":"0","promo_item":"AVpfMVD-ilAPnD_xW6bu","promo_qty":"3"}
    ]
```


> Three most important function
- from_json - string -> structured column
- explode - row -> multiple rows
- explode_outer -> Same as explode but handle if data is NULL

## Ordered Products

In [0]:
df.display()

In [0]:
df.select("order_number","ordered_products").display()

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col

ordered_product_schema = ArrayType(
    StructType([
        StructField("curr", StringType()),
        StructField("id", StringType()),
        StructField("name", StringType()),
        StructField("price", StringType()),
        StructField("promotion_info", StringType()),
        StructField("qty", StringType()),
        StructField("unit", StringType())
    ])
)

df_parsed = df.withColumn("ordered_products", from_json(col("ordered_products"), ordered_product_schema ))



In [0]:
df_parsed.printSchema()

In [0]:
from pyspark.sql.functions import explode_outer

df_exploded_op = df_parsed.withColumn("ordered_products", explode_outer("ordered_products"))

df_exploded_op.display()

In [0]:
df_ordered_products = df_exploded_op.select(
    "customer_id",
    "customer_name",
    "order_number",
    "ordered_products.id",
    col("ordered_products.name").alias("product_name"),
    "ordered_products.price",
    "ordered_products.curr",
    "ordered_products.qty",
    "ordered_products.unit"
    )

df_ordered_products.display()

In [0]:
'''

1. Read the column
2. define schema in structfield and structtype
3. use from json and parse the schema
4. explode it 
5. select the columns
6. combine the code 
7. write it to silver schema

'''

In [0]:
df = spark.read.table("ecommerce_analytics.bronze.sales_orders")

display(df.select("clicked_items"))

In [0]:
df.printSchema()

In [0]:
# Step 1 - Define Schema
from pyspark.sql.types import *

clicked_item_schema = ArrayType(
    ArrayType(StringType())
)

In [0]:
# Step 2 -> Parsing the schema through column
from pyspark.sql.functions import from_json, col
df_parsed = df.withColumn("clicked_items", from_json(col("clicked_items"), clicked_item_schema))
df_parsed.printSchema()

In [0]:
df_parsed.display()

In [0]:
# Step 3 -> Explode the column into multiple rows

from pyspark.sql.functions import explode_outer

df_exploded = df_parsed.withColumn("clicked_items", explode_outer("clicked_items"))
df_exploded.display()


In [0]:
'''
index 0 -> product_id
index 1 -> score
'''

clicked_items = df_exploded.select(
    "customer_id",
    "customer_name",
    "order_number",
    col("clicked_items")[0].alias("product_id"),
    col("clicked_items")[1].alias("score")
)
clicked_items.display()

In [0]:
df = spark.read.table("ecommerce_analytics.bronze.sales_orders")
df.display()

In [0]:
from pyspark.sql.functions import from_unixtime

def sales_orders(df):
    sales_orders = df.select(
        "order_number",
        "customer_id",
        "customer_name",
        "number_of_line_items",
        from_unixtime(col("order_datetime")).alias("order_timestamp")
    )
    return sales_orders

